<a href="https://colab.research.google.com/github/rist-kobe/HPC-Programming/blob/main/Tuning/sample_code/03_prof-ex/03_prof-ex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup

Install GNU Fortran and NVIDIA HPC SDK (optional for C-only examples; can take ~30 min)

In [ ]:
!sudo apt-get update -y
!sudo apt-get install -y build-essential gfortran curl gnupg

# Optional: install NVIDIA HPC SDK (large download, not needed for C-only examples).
# Uncomment the following lines to install it.
#!curl -fsSL https://developer.download.nvidia.com/hpc-sdk/ubuntu/DEB-GPG-KEY-NVIDIA-HPC-SDK | sudo gpg --dearmor -o /usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg
#!echo 'deb [signed-by=/usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg] https://developer.download.nvidia.com/hpc-sdk/ubuntu/amd64 /' | sudo tee /etc/apt/sources.list.d/nvhpc.list
#!sudo apt-get update -y
#!sudo apt-get install -y nvhpc-22-7-cuda-multi

import glob, os
nvhpc_bins = sorted(glob.glob('/opt/nvidia/hpc_sdk/Linux_x86_64/*/compilers/bin'), key=lambda p: tuple(int(x) for x in p.split('/Linux_x86_64/')[1].split('/')[0].replace('-', '.').split('.')), reverse=True)
if nvhpc_bins:
    nvhpc_bin = nvhpc_bins[0]
    current_path = os.environ.get('PATH', '')
    if nvhpc_bin not in current_path.split(':'):
        os.environ['PATH'] = nvhpc_bin + (':' + current_path if current_path else '')
    print('Using NVIDIA HPC SDK:', nvhpc_bin)
else:
    print('NVIDIA HPC SDK not found (optional; not needed for C-only examples).')


Clone the repository and change to the `03_prof-ex` directory.

In [ ]:
%cd /content
!rm -rf HPC-Programming
!git clone https://github.com/rist-kobe/HPC-Programming.git
%cd HPC-Programming/Tuning/sample_code/03_prof-ex
!ls


# Exercise on a performance analysis with gprof (and perf)
* Author:   Yukihiro Ota (yota@rist.or.jp)
* Last update: 26th Jan. 2024

## Preparation
First, you need to obtain Mersenne-Twister source code for a random-number generator. This sample **DOES NOT** include it.
### Fortran
* Visit the following website and obtain `mtfort90.f`: [http://www.math.sci.hiroshima-u.ac.jp/m-mat/MT/VERSIONS/FORTRAN/fortran.html](http://www.math.sci.hiroshima-u.ac.jp/m-mat/MT/VERSIONS/FORTRAN/fortran.html)
* Extract module part, `mtmod`, from `mtfort90.f`. Then the resultant code is named by `mtfort90.f90`.
* Copy `mtfort90.f90` in `src/fortran`.
### C
* Visit the following website and obtain `mt19937ar.c` and `mt19937ar.h`: [http://www.math.sci.hiroshima-u.ac.jp/m-mat/MT/mt.html](http://www.math.sci.hiroshima-u.ac.jp/m-mat/MT/mt.html) and [http://www.math.sci.hiroshima-u.ac.jp/m-mat/MT/MT2002/mt19937ar.html](http://www.math.sci.hiroshima-u.ac.jp/m-mat/MT/MT2002/mt19937ar.html). Use of `mt19937ar.sep.tgz` is recommended.
* Copy `mt19937ar.c` and `mt19937ar.h` in `src/c`.
### On Mersenne-Twister 
* M. Matsumoto and T. Nishimura, "Mersenne Twister: A 623-Dimensionally Equidistributed Uniform Pseudo-Random Number Generator", ACM Transactions on Modeling and Computer Simulation, Vol. 8, No. 1, January 1998, pp 3-30.
* [Mersenne Twister Home Page](http://www.math.sci.hiroshima-u.ac.jp/m-mat/MT/mt.html)

## Instruction: Compile
1. Source code is stored in `src/`. Choose either fortran or c.
2. Change directory

In [ ]:
!cd src/c # On c


3. Make

In [ ]:
!make


The code is successfully compiled by:
   * GNU (8.5.0) on x86-64 systems

## Instruction: Run and do a performance analysis
1. Sample scripts are stored in `tests/`. Choose either fortran or c.
2. Change directory

In [ ]:
!cd tests/c # On c


3. Run a job script, `run.sh`.

In [ ]:
# One example
!bash run.sh
# Another example
!chmod 755 run.sh
!./run.sh


4. The result of `gprof` will be summarized in files if setting `-pg` in compiler's options.
5. You can also try to use another profiler, `perf`, according to the information form Linux kernel. You move to directory `tests/perf`.

## Exercise
1. Find functions corresponding to hotspot. 
2. Change the optimization level or a kind of compiler. What about the results? 
3. Make a timer routine. Insert it in any place of the source code.  Then, measure elapsed time. Also, find which lines in the hotspot functions have high runtime costs.  
4. Visualization: This is NOT related to performance tuning. You can visualize the resultant data, `trj.xyz`, with `Jmol`, for example: [http://jmol.sourceforge.net](http://jmol.sourceforge.net)
 
## Advanced topics
1. Consider how to optimize this code, according to the performance analysis and compiler's report.
2. How do you evaluate the used memory in runtime? A simple way on a Linux system is to read `/proc/self/status`. 
3. Measure FLOP/s and memory band width in a specific (small) part of this code, by hand. Compare them to their ideal values.  To know an effective memory bandwidth, use of `STREAM` would be useful: [http://www.cs.virginia.edu/stream](http://www.cs.virginia.edu/stream)
4. Understanding **bottleneck** is often important: Why is your program is so slow? In this case, a performance analysis model would be useful. Try the Roofline model. What kind of knowledge can you obtain via this approach?